# 当当网Python书籍爬取与数据清洗

本Notebook用于爬取当当网Python书籍数据，并进行数据清洗与保存。

## 设计思路
1. 设置爬虫目录，确保所有数据和代码都在该目录下运行。
2. 导入必要库，包含网页请求、解析和数据处理。
3. 编写爬虫代码，分为搜索页爬取、详情页解析、数据清洗。
4. 保存爬取和清洗后的数据到指定目录。

每一步核心逻辑均配有详细中文注释，便于理解和后续维护。

In [ ]:
# 设置爬虫目录，确保所有操作都在该目录下进行
import os

# 获取当前脚本所在目录
SCRIPT_DIR = os.path.dirname(os.path.abspath("run_scraper.ipynb"))
# 项目根目录
PROJECT_ROOT = os.path.dirname(os.path.dirname(SCRIPT_DIR))

# 定义数据目录
DATA_RAW_DIR = os.path.join(PROJECT_ROOT, 'data_raw')
DATA_CLEAN_DIR = os.path.join(PROJECT_ROOT, 'data_clean')
OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'output')

# 创建目录（如不存在则自动创建）
for d in [DATA_RAW_DIR, DATA_CLEAN_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

print("爬虫目录及数据目录已设置：", SCRIPT_DIR)

爬虫目录及数据目录已设置： /Users/kaka/Documents/中大岭院/课程/数据分析与决策/Proj_homework/code/爬虫


In [ ]:
# 导入必要库
import requests  # 用于发送网页请求
from bs4 import BeautifulSoup  # 用于解析网页内容
import pandas as pd  # 用于数据处理
import time  # 控制请求间隔，防止被封IP
import random  # 随机延时
import re  # 正则表达式，提取信息

print("已导入所有爬虫和数据处理相关库")

已导入所有爬虫和数据处理相关库


In [ ]:
# 配置当当网搜索页面请求参数
BASE_URL = "https://search.dangdang.com/"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "zh-CN,zh;q=0.9,en;q=0.8",
    "Accept-Encoding": "gzip, deflate, br",
    "Referer": "https://www.dangdang.com/",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
}

print("请求头和基础URL已配置")

请求头和基础URL已配置


In [ ]:
# 定义获取搜索页HTML的函数，带重试和延时

def fetch_search_page(page, retries=3, pause=(1.0, 2.0)):
    """获取特定页的搜索结果 HTML，带重试和延时。"""
    # 构造请求URL
    if page == 1:
        url = "https://search.dangdang.com/?key=python&act=input"
    else:
        url = f"https://search.dangdang.com/?key=python&act=input&page_index={page}"
    print(f"请求第 {page} 页，URL：{url}")

    for attempt in range(1, retries + 1):
        try:
            response = requests.get(url, headers=HEADERS, timeout=10)
            print(f"尝试 {attempt}：状态码 {response.status_code}")
            response.encoding = "gbk"
            if response.status_code == 200:
                print("✅ 解析HTML...")
                return BeautifulSoup(response.text, 'html.parser')
            time.sleep(pause[0])
        except Exception as e:
            print(f"尝试 {attempt} 异常：{e}")
            time.sleep(pause[0])
    return None

print("定义了搜索页请求函数 fetch_search_page")

定义了搜索页请求函数 fetch_search_page


In [ ]:
# 定义解析搜索页书籍条目的函数

def parse_book_items(soup):
    """解析搜索页中的书籍条目。"""
    if not soup:
        return []

    # 查找所有书籍条目
    items = soup.find_all('li', class_=lambda x: x and 'line' in x and 'search_batch' not in x)
    books = []

    for item in items:
        book = {}
        # 提取书名
        title_elem = item.find('a', class_='pic')
        book['书名'] = title_elem.get('title', '未知') if title_elem else '未知'
        # 提取详情链接
        if title_elem:
            detail_url = title_elem.get('href')
            if detail_url and not detail_url.startswith('http'):
                detail_url = 'https:' + detail_url
            book['详情链接'] = detail_url
        # 提取作者
        author_elem = item.find('p', class_='search_book_author')
        if author_elem:
            author_link = author_elem.find('span').find('a') if author_elem.find('span') else None
            book['作者'] = author_link.get_text().strip() if author_link else '未知'
        # 提取出版信息
        publish_spans = author_elem.find_all('span') if author_elem else []
        if len(publish_spans) >= 2:
            publish_text = publish_spans[1].get_text().strip()
            year_match = re.search(r'(\d{4})', publish_text)
            if year_match:
                book['出版年份'] = year_match.group(1)
        if len(publish_spans) >= 3:
            publisher_link = publish_spans[2].find('a')
            if publisher_link:
                book['出版社'] = publisher_link.get_text().strip()
            else:
                book['出版社'] = publish_spans[2].get_text().strip()
        # 提取评论数
        comment_elem = item.find('a', class_='search_comment_num')
        if comment_elem:
            comment_text = comment_elem.get_text().replace('条评论', '').strip()
            book['评论数'] = comment_text or '0'
        # 提取价格
        pre_price = item.find('span', class_='search_pre_price')
        if pre_price:
            book['原价(元)'] = pre_price.get_text().replace('¥', '').strip()
        now_price = item.find('span', class_='search_now_price')
        if now_price:
            book['折后价(元)'] = now_price.get_text().replace('¥', '').strip()
        books.append(book)
    return books

print("定义了搜索页解析函数 parse_book_items")

定义了搜索页解析函数 parse_book_items


In [ ]:
# 定义主爬虫函数，循环爬取多页数据

def scrape_dangdang_books(pages=3):
    """爬取多页当当Python图书搜索结果。"""
    all_books = []
    for p in range(1, pages + 1):
        print(f"正在爬取第 {p} 页...")
        soup = fetch_search_page(p)
        if soup is None:
            print(f"第 {p} 页请求失败")
            continue
        page_books = parse_book_items(soup)
        all_books.extend(page_books)
        print(f"第 {p} 页获取到 {len(page_books)} 条记录")
        if p < pages:
            time.sleep(random.uniform(1, 2))  # 随机延时，防止被封IP
    return pd.DataFrame(all_books)

print("定义了主爬虫函数 scrape_dangdang_books")

定义了主爬虫函数 scrape_dangdang_books


In [ ]:
# 数据清洗函数：去重、类型转换、计算折扣率

def clean_books(df):
    """清洗数据并生成可分析的数据框。"""
    if df is None or len(df) == 0:
        print("警告：原始数据为空，返回空 DataFrame")
        return pd.DataFrame(columns=['书名', '详情链接', '作者', '出版年份', '出版社', '评论数', '原价(元)', '折后价(元)', '折扣率'])
    df = df.copy()
    df.replace({"未知": None, "": None}, inplace=True)
    df.drop_duplicates(subset=["书名"], inplace=True)
    # 确保必要的列存在
    required_cols = ['评论数', '原价(元)', '折后价(元)', '出版年份']
    for col in required_cols:
        if col not in df.columns:
            df[col] = None
    df["评论数"] = pd.to_numeric(df["评论数"], errors="coerce").fillna(0).astype(int)
    df["原价(元)"] = pd.to_numeric(df["原价(元)"], errors="coerce")
    df["折后价(元)"] = pd.to_numeric(df["折后价(元)"], errors="coerce")
    df["出版年份"] = pd.to_numeric(df["出版年份"], errors="coerce").astype("Int64")
    df["折扣率"] = (df["折后价(元)"] / df["原价(元)"]).round(2)
    return df

print("定义了数据清洗函数 clean_books")

定义了数据清洗函数 clean_books


In [ ]:
# 保存清洗后的数据到指定目录

def save_outputs(df_clean):
    """保存清洗后的数据到 data_raw 目录。"""
    raw_path = os.path.join(DATA_RAW_DIR, 'python_books_raw.csv')
    report_path = os.path.join(OUTPUT_DIR, 'data_summary.txt')
    print(f"保存到：{raw_path}")
    df_clean.to_csv(raw_path, index=False, encoding="utf-8-sig")
    with open(report_path, "w", encoding="utf-8") as f:
        f.write("Python书籍爬取摘要\n")
        f.write(f"清洗后记录数：{len(df_clean)}\n")
        if len(df_clean) > 0:
            f.write(f"平均折后价：{df_clean['折后价(元)'].mean():.2f} 元\n")
            f.write(f"平均折扣率：{df_clean['折扣率'].mean():.2f}\n")
    print(f"数据已保存：{raw_path}, {report_path}")
    return raw_path, report_path

print("定义了数据保存函数 save_outputs")

定义了数据保存函数 save_outputs


In [ ]:
# 主流程：爬取、清洗、保存

print("=" * 50)
print("开始爬取当当网Python书籍数据...")
print("=" * 50)

# 1. 爬取数据
print("\n[1/3] 执行数据爬取...")
df_raw = scrape_dangdang_books(pages=3)
if len(df_raw) == 0:
    print("警告：未能获取到任何数据！")

# 2. 数据清洗
print("\n[2/3] 执行数据清洗...")
df_clean = clean_books(df_raw)
print(f"清洗后共 {len(df_clean)} 条记录")

# 3. 保存清洗后的数据到 data_raw
print("\n[3/3] 保存数据...")
save_outputs(df_clean)

# 完成
print("\n完成！")
print("=" * 50)
print(f"最终结果：共获取 {len(df_clean)} 条记录")
print("=" * 50)

开始爬取当当网Python书籍数据...

[1/3] 执行数据爬取...
正在爬取第 1 页...
请求第 1 页，URL：https://search.dangdang.com/?key=python&act=input
尝试 1：状态码 200
✅ 解析HTML...
第 1 页获取到 60 条记录
正在爬取第 2 页...
请求第 2 页，URL：https://search.dangdang.com/?key=python&act=input&page_index=2
尝试 1：状态码 200
✅ 解析HTML...
第 2 页获取到 60 条记录
正在爬取第 3 页...
请求第 3 页，URL：https://search.dangdang.com/?key=python&act=input&page_index=3
尝试 1：状态码 200
✅ 解析HTML...
第 3 页获取到 60 条记录

[2/3] 执行数据清洗...
清洗后共 154 条记录

[3/3] 保存数据...
保存到：/Users/kaka/Documents/中大岭院/课程/数据分析与决策/Proj_homework/data_raw/python_books_raw.csv
数据已保存：/Users/kaka/Documents/中大岭院/课程/数据分析与决策/Proj_homework/data_raw/python_books_raw.csv, /Users/kaka/Documents/中大岭院/课程/数据分析与决策/Proj_homework/output/data_summary.txt

完成！
最终结果：共获取 154 条记录
